# Audit dataset Amazon Reviews'23

Notebook này chạy dataset gate đầu tiên cho hướng GRAPES-to-recommendation đã khóa. Notebook chỉ tải official pure-ID 0-core rating-only artifact, tính provenance và graph statistic, rồi ghi JSON summary. Notebook không train model và không tự chọn rating threshold.

Chạy `All_Beauty` trước để validate pipeline, sau đó chạy cùng cell cho `Baby_Products`. Lưu output vào persistent storage.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

CODE_ROOT = Path('/content/Do An/06_code')
assert CODE_ROOT.is_dir(), f'Hãy copy 06_code tới {CODE_ROOT} trước khi chạy.'
SCRIPT = CODE_ROOT / 'scripts' / 'analyze_amazon_dataset.py'
CACHE_ROOT = Path('/content/amazon_reviews_2023_cache')
OUTPUT_ROOT = CODE_ROOT / 'audit_outputs'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SOURCES = {
    'All_Beauty': 'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/benchmark/0core/rating_only/All_Beauty.csv.gz',
    'Baby_Products': 'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/benchmark/0core/rating_only/Baby_Products.csv.gz',
}
CATEGORY = 'All_Beauty'
ARTIFACT_URL = SOURCES[CATEGORY]
OUTPUT_PATH = OUTPUT_ROOT / f'{CATEGORY}_dataset_audit.json'
print({'category': CATEGORY, 'artifact': ARTIFACT_URL, 'output': str(OUTPUT_PATH)})


In [ ]:
command = [
    sys.executable, str(SCRIPT),
    '--input', ARTIFACT_URL,
    '--source', ARTIFACT_URL,
    '--cache-dir', str(CACHE_ROOT),
    '--output', str(OUTPUT_PATH),
    '--pair-audit', 'sqlite',
    '--timestamp-audit',
]
subprocess.run(command, check=True)
audit = json.loads(OUTPUT_PATH.read_text(encoding='utf-8'))
print(json.dumps({
    'sha256': audit['sha256'],
    'compressed_bytes': audit['compressed_bytes'],
    'row_count': audit['row_count'],
    'valid_row_count': audit['valid_row_count'],
    'unique_users': audit['unique_users'],
    'unique_items': audit['unique_items'],
    'duplicate_user_item_rows': audit['duplicate_user_item_rows'],
    'invalid_or_missing': audit['invalid_or_missing'],
    'candidate_absolute_split': audit['candidate_absolute_split'],
}, indent=2))


## Ranh giới diễn giải

JSON là evidence do project tạo ra từ exact downloaded bytes. Phải tách provider-published count, project-derived count, protocol choice đang đề xuất và giá trị unknown. Chưa bắt đầu training cho đến khi checksum, duplicate policy, implicit-positive rule, temporal split, warm-start exclusion và negative-sampling policy được ghi vào dataset audit và hai continuity file.